Reliability Ratios Calculation - Python Implementation
Replicates Stata do-file: Realiability Ratios_Table5&20_definitive.do

In [19]:
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from statsmodels.regression.mixed_linear_model import MixedLM

warnings.filterwarnings("ignore")

0. ENVIRONMENT SETUP

In [20]:
import os
import platform

# Auto-detect WSL vs native Windows and set data path accordingly
_raw_path = (
    "C:/Users/Ishmail Baako/Documents/reliability ratios/data/standardized_long.dta"
)

if "microsoft" in platform.uname().release.lower() or os.path.exists("/mnt/c"):
    # Running in WSL — convert Windows path to WSL path
    DATA_PATH = "/mnt/c" + _raw_path[2:].replace("\\", "/")
else:
    DATA_PATH = _raw_path

print(f"Data path: {DATA_PATH}")

Data path: C:/Users/Ishmail Baako/Documents/reliability ratios/data/standardized_long.dta


1. LOAD DATA

In [21]:
# Load Stata file directly
df = pd.read_stata(DATA_PATH)

print(f"Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")

df.head()

Data loaded: 10732 rows, 138 columns


,survey_id,hh_id,survey,hh_id_uniq,outcome,foid,aoid,cluster_id,wave,country,...,outcome_signal_w98,outcome_signal_w99,outcome_signal_w100,enum_id,enum,hh_obs,enum_id_1,hh_size_dum,land_dum,crop_dum
0,p001_02,10-009,0,p001_02_10-009,-0.763948,112,13,10,02,PHL,...,-0.763948,-0.763948,-0.763948,p001_02_b13,p001_02_b13,4,p001_02_s112,1.0,0.0,0.0
1,p001_02,10-009,1,p001_02_10-009,-0.763948,112,13,10,02,PHL,...,-0.763948,-0.763948,-0.763948,p001_02_s112,p001_02_s112,4,p001_02_s112,1.0,0.0,0.0
2,p001_02,10-009,0,p001_02_10-009,-0.650055,112,13,10,02,PHL,...,-0.719314,-0.720021,-0.720727,p001_02_b13,p001_02_b13,4,p001_02_s112,0.0,0.0,0.0
3,p001_02,10-009,1,p001_02_10-009,-0.720727,112,13,10,02,PHL,...,-0.719314,-0.720021,-0.720727,p001_02_s112,p001_02_s112,4,p001_02_s112,0.0,0.0,0.0
4,p001_02,10-010,0,p001_02_10-010,-0.008197,111,13,10,02,PHL,...,-0.255075,-0.257595,-0.260114,p001_02_b13,p001_02_b13,4,p001_02_s111,1.0,0.0,0.0


2. DEFINE VARIABLES AND LABELS

In [22]:
# Survey sites
sites = ["p011_01", "p011_02", "p008_01", "p001_02"]

# Site labels
site_labels = {
    "p001_02": "PHL ICM endline",
    "p008_01": "MLI ATAI baseline",
    "p011_01": "BFA UPPN baseline",
    "p011_02": "BFA UPPN midline",
}

# Mapping from numeric pair IDs (used in code logic) to actual categorical values
# The Stata file stores pair as categorical strings, not integers
pair_values = {
    1: "HH Size",
    2: "Land",
    3: "No. of Crops",
    4: "Asset index",
}

# Outcome variable labels (pairs 1-4, plus pooled=5)
outcome_labels = {
    1: "Household size",
    2: "Land size (ha)",
    3: "No. of crop types",
    4: "Asset index",
    5: "Pooled outcomes",
}

# Household head characteristics
hhhead_chars = ["hhhead_age", "hhhead_female", "hhhead_educ"]
hhhead_chars_mi = ["mi_hhhead_age", "mi_hhhead_female", "mi_hhhead_educ"]

# Weights for truth models
weights = [100, 90, 50, 10, 0]

3. DATA PREPARATION

In [23]:
# Create dummy variables for pair using actual categorical values
df["dummy1"] = (df["pair"] == pair_values[1]).astype(int)
df["dummy2"] = (df["pair"] == pair_values[2]).astype(int)
df["dummy3"] = (df["pair"] == pair_values[3]).astype(int)

# Ensure enum is string type for grouping
df["enum"] = df["enum"].astype(str)

# Ensure num_proj_id is categorical for C() wrapper
if df["num_proj_id"].dtype in ["int64", "float64"]:
    df["num_proj_id"] = df["num_proj_id"].astype(str)

# Check for missing values and create indicators if needed
print("\nChecking for missing values in key variables:")
for col in ["hhhead_age", "hhhead_female", "hhhead_educ"]:
    n_missing = df[col].isna().sum()
    print(f"  {col}: {n_missing} missing ({n_missing / len(df) * 100:.2f}%)")

# Fill missing values with median/mode to avoid issues
df["hhhead_age"] = df["hhhead_age"].fillna(df["hhhead_age"].median())
df["hhhead_female"] = df["hhhead_female"].fillna(
    df["hhhead_female"].mode()[0] if len(df["hhhead_female"].mode()) > 0 else 0
)

# Handle education - convert to string first, then fillna, then back to category
if df["hhhead_educ"].dtype.name == "category":
    df["hhhead_educ"] = df["hhhead_educ"].astype(str)

df["hhhead_educ"] = df["hhhead_educ"].fillna("Missing")

# Clean any special characters in the categories
df["hhhead_educ"] = (
    df["hhhead_educ"]
    .str.replace(" ", "_")
    .str.replace("[^A-Za-z0-9_]+", "_", regex=True)
)

# Convert back to category
df["hhhead_educ"] = df["hhhead_educ"].astype("category")

print("\nData preparation complete")
print(f"  - Enum groups: {df['enum'].nunique()}")
print(f"  - Project IDs: {df['num_proj_id'].nunique()}")
print(f"  - Education categories: {df['hhhead_educ'].nunique()}")
print(f"  - Dummy1 (HH Size) sum: {df['dummy1'].sum()}")
print(f"  - Dummy2 (Land) sum: {df['dummy2'].sum()}")
print(f"  - Dummy3 (No. of Crops) sum: {df['dummy3'].sum()}")


Checking for missing values in key variables:
  hhhead_age: 0 missing (0.00%)
  hhhead_female: 0 missing (0.00%)
  hhhead_educ: 0 missing (0.00%)

Data preparation complete
  - Enum groups: 276
  - Project IDs: 4
  - Education categories: 21
  - Dummy1 (HH Size) sum: 3928
  - Dummy2 (Land) sum: 2636
  - Dummy3 (No. of Crops) sum: 2602


4. MIXED EFFECTS MODELS FOR TRUTH ESTIMATION

In [24]:
print("\n" + "=" * 80)
print("STEP 1: Estimating Truth Models (w100 and w0)")
print("=" * 80)

# Build formula for fixed effects
dummy_cols = ["dummy1", "dummy2", "dummy3"]
continuous_vars = ["hhhead_age", "hhhead_female"]

# Full formula (with dummies) for pooled models
formula_parts = continuous_vars + dummy_cols
formula_parts.append("C(hhhead_educ)")
formula_parts.append("C(num_proj_id)")
formula_fixed = " + ".join(formula_parts)

# Reduced formula (without dummies) for individual outcome models
formula_parts_no_dummies = continuous_vars.copy()
formula_parts_no_dummies.append("C(hhhead_educ)")
formula_parts_no_dummies.append("C(num_proj_id)")
formula_fixed_no_dummies = " + ".join(formula_parts_no_dummies)

print(f"\nFull formula: outcome ~ {formula_fixed}")
print(f"Reduced formula (individual outcomes): outcome ~ {formula_fixed_no_dummies}")


def safe_resid(result):
    """Extract residuals, falling back to fixed-effects-only if random effects
    are singular.
    """
    try:
        return result.resid
    except ValueError:
        # Singular covariance — compute residuals from fixed effects only
        return result.model.endog - np.dot(result.model.exog, result.fe_params)


# Model 1: Truth based on 100% survey weight
print("\nFitting mixed model for outcome_signal_w100...")
try:
    model_df = df.dropna(
        subset=[
            "outcome_signal_w100",
            "hhhead_age",
            "hhhead_female",
            "hhhead_educ",
            "num_proj_id",
            "enum",
        ]
        + dummy_cols
    )
    print(f"  Sample size: {len(model_df)} observations")

    model1 = MixedLM.from_formula(
        f"outcome_signal_w100 ~ {formula_fixed}",
        data=model_df,
        groups=model_df["enum"],
        re_formula="1",
    )
    result1 = model1.fit(reml=True, method=["lbfgs", "bfgs"])
    print(f"  Model 1 converged: {result1.converged}")
    print(f"  Log-likelihood: {result1.llf:.2f}")

    residuals = pd.Series(safe_resid(result1), index=model_df.index)
    df["outcome_truth_resid"] = residuals.reindex(df.index).fillna(0)

except Exception as e:
    print(f"  ERROR fitting model 1: {e}")
    print("\n  Trying simplified model without education variable...")
    try:
        formula_simple = " + ".join(continuous_vars + dummy_cols + ["C(num_proj_id)"])
        model1 = MixedLM.from_formula(
            f"outcome_signal_w100 ~ {formula_simple}",
            data=model_df,
            groups=model_df["enum"],
            re_formula="1",
        )
        result1 = model1.fit(reml=True, method=["lbfgs", "bfgs"])
        print(f"  Simplified model converged: {result1.converged}")
        residuals = pd.Series(safe_resid(result1), index=model_df.index)
        df["outcome_truth_resid"] = residuals.reindex(df.index).fillna(0)
        formula_fixed = formula_simple
    except Exception as e2:
        print(f"  ERROR with simplified model: {e2}")
        raise

# Model 2: Truth based on 0% survey weight (100% backcheck)
print("\nFitting mixed model for outcome_signal_w0...")
try:
    model_df = df.dropna(
        subset=[
            "outcome_signal_w0",
            "hhhead_age",
            "hhhead_female",
            "hhhead_educ",
            "num_proj_id",
            "enum",
        ]
        + dummy_cols
    )
    print(f"  Sample size: {len(model_df)} observations")

    model2 = MixedLM.from_formula(
        f"outcome_signal_w0 ~ {formula_fixed}",
        data=model_df,
        groups=model_df["enum"],
        re_formula="1",
    )
    result2 = model2.fit(reml=True, method=["lbfgs", "bfgs"])
    print(f"  Model 2 converged: {result2.converged}")
    print(f"  Log-likelihood: {result2.llf:.2f}")

    residuals = pd.Series(safe_resid(result2), index=model_df.index)
    df["outcome_bctruth_resid"] = residuals.reindex(df.index).fillna(0)

except Exception as e:
    print(f"  ERROR fitting model 2: {e}")
    raise

print("\nTruth models fitted successfully!")


STEP 1: Estimating Truth Models (w100 and w0)

Full formula: outcome ~ hhhead_age + hhhead_female + dummy1 + dummy2 + dummy3 + C(hhhead_educ) + C(num_proj_id)
Reduced formula (individual outcomes): outcome ~ hhhead_age + hhhead_female + C(hhhead_educ) + C(num_proj_id)

Fitting mixed model for outcome_signal_w100...
  Sample size: 10732 observations
  Model 1 converged: True
  Log-likelihood: inf

Fitting mixed model for outcome_signal_w0...
  Sample size: 10732 observations
  Model 2 converged: True
  Log-likelihood: inf

Truth models fitted successfully!


5. INITIALIZE RESULTS STORAGE

In [25]:
results_list = []
means_list = []

6. LOOP OVER OUTCOMES AND WEIGHTS

In [26]:
print("\n" + "=" * 80)
print("STEP 2: Calculating Reliability Ratios")
print("=" * 80)

for k in range(1, 6):  # Loop over outcomes 1-5
    print(f"\n--- Processing Outcome {k}: {outcome_labels[k]} ---")

    # Use full formula with dummies for pooled (k=5), reduced formula for individual
    current_formula = formula_fixed if k == 5 else formula_fixed_no_dummies

    for _, i in enumerate(weights):
        j = 100 - i  # Inverse weight

        print(f"  Weight combination: ({i / 100:.1f}, {j / 100:.1f})")

        # Determine if this is pooled outcomes (k=5) or individual outcome
        if k == 5:
            # Pooled outcomes: use all data
            mask_survey = df["survey"] == 1
            mask_backcheck = df["survey"] == 0
            mask_all = pd.Series([True] * len(df), index=df.index)
        else:
            # Individual outcome: filter by pair using categorical values
            pair_val = pair_values[k]
            mask_survey = (df["survey"] == 1) & (df["pair"] == pair_val)
            mask_backcheck = (df["survey"] == 0) & (df["pair"] == pair_val)
            mask_all = df["pair"] == pair_val

        # ----------------------------------------------------------------
        # Calculate basic statistics
        # ----------------------------------------------------------------

        # Variance and mean of Y: survey
        var_s = df.loc[mask_survey, "outcome"].var()
        mu_s = df.loc[mask_survey, "outcome"].mean()
        N_survey = mask_survey.sum()

        # Variance and mean of Y: backcheck
        var_b = df.loc[mask_backcheck, "outcome"].var()
        mu_b = df.loc[mask_backcheck, "outcome"].mean()

        # Variance and mean of signal
        signal_col = f"outcome_signal_w{i}"
        var_signal = df.loc[mask_survey, signal_col].var()
        mu_signal = df.loc[mask_survey, signal_col].mean()

        # ----------------------------------------------------------------
        # Run mixed effects regression for this weight
        # ----------------------------------------------------------------

        df_subset = df[mask_all].copy()
        # Drop missing values
        drop_cols = [
            signal_col,
            "hhhead_age",
            "hhhead_female",
            "hhhead_educ",
            "num_proj_id",
            "enum",
        ]
        if k == 5:
            drop_cols += dummy_cols
        df_subset = df_subset.dropna(subset=drop_cols)

        # Ensure categorical variables are preserved
        df_subset["enum"] = df_subset["enum"].astype(str)
        df_subset["num_proj_id"] = df_subset["num_proj_id"].astype(str)
        if df_subset["hhhead_educ"].dtype != "category":
            df_subset["hhhead_educ"] = df_subset["hhhead_educ"].astype("category")

        try:
            model = MixedLM.from_formula(
                f"{signal_col} ~ {current_formula}",
                data=df_subset,
                groups=df_subset["enum"],
                re_formula="1",
            )
            result = model.fit(reml=True, method=["lbfgs", "bfgs"])

            if not result.converged:
                print(f"    WARNING: Model did not converge for k={k}, i={i}")

            # Extract residuals safely
            outcome_resid = safe_resid(result)

            # Variance of residuals
            sigma_resid = outcome_resid.var()

            # Extract random effect variance (enumerator level)
            sigma_enum = result.cov_re.iloc[0, 0]

        except Exception as e:
            print(f"    ERROR fitting model for k={k}, i={i}: {e}")
            continue

        # ----------------------------------------------------------------
        # Calculate measurement error variances
        # ----------------------------------------------------------------

        # Create temporary residual columns - align with df_subset index
        df_subset.loc[:, "outcome_resid"] = (
            outcome_resid
            if isinstance(outcome_resid, np.ndarray)
            else outcome_resid.values
        )

        # Get the corresponding truth residuals for this subset
        truth_resid_subset = df.loc[df_subset.index, "outcome_truth_resid"]
        bctruth_resid_subset = df.loc[df_subset.index, "outcome_bctruth_resid"]

        # Measurement error: survey
        df_subset["outcome_resid1"] = (
            truth_resid_subset.values - df_subset["outcome_resid"].values
        )
        var_s_me = df_subset.loc[df_subset["survey"] == 1, "outcome_resid1"].var()
        mu_s_me = df_subset.loc[df_subset["survey"] == 1, "outcome_resid1"].mean()

        # Measurement error: backcheck
        df_subset["outcome_resid2"] = (
            bctruth_resid_subset.values - df_subset["outcome_resid"].values
        )
        var_b_me = df_subset.loc[df_subset["survey"] == 0, "outcome_resid2"].var()
        mu_b_me = df_subset.loc[df_subset["survey"] == 0, "outcome_resid2"].mean()

        # ----------------------------------------------------------------
        # Calculate reliability ratios
        # ----------------------------------------------------------------

        # RR_KY: Classical test theory formula
        rr_ky_s = var_signal / (var_signal + var_s_me)
        rr_ky_b = var_signal / (var_signal + var_b_me)

        # RR_AS: Abowd & Stinson formula
        rr_as_s = 1 - (var_s_me / var_s)
        rr_as_b = 1 - (var_b_me / var_b)

        # IIC: Intra-class correlation
        iic = sigma_enum / (sigma_enum + sigma_resid)

        # ----------------------------------------------------------------
        # Calculate sample sizes by site
        # ----------------------------------------------------------------

        N_total = N_survey
        site_counts = {}
        for site in sites:
            if k == 5:
                count = (df["survey_id"] == site).sum() / 2
            else:
                count = ((df["survey_id"] == site) & (df["pair"] == pair_val)).sum() / 2
            site_counts[site] = int(count)

        # ----------------------------------------------------------------
        # Store results
        # ----------------------------------------------------------------

        result_row = {
            "outcome": outcome_labels[k],
            "weight_s": i / 100,
            "weight_b": j / 100,
            "var_s": var_s,
            "var_b": var_b,
            "var_signal": var_signal,
            "var_s_me": var_s_me,
            "var_b_me": var_b_me,
            "rr_ky_s": rr_ky_s,
            "rr_ky_b": rr_ky_b,
            "rr_as_s": rr_as_s,
            "rr_as_b": rr_as_b,
            "iic": iic,
            "N": N_total,
            "N_p001_02": site_counts["p001_02"],
            "N_p008_01": site_counts["p008_01"],
            "N_p011_01": site_counts["p011_01"],
            "N_p011_02": site_counts["p011_02"],
        }
        results_list.append(result_row)

        mean_row = {
            "outcome": outcome_labels[k],
            "weight_s": i / 100,
            "weight_b": j / 100,
            "mu_s": mu_s,
            "mu_b": mu_b,
            "mu_signal": mu_signal,
            "mu_s_me": mu_s_me,
            "mu_b_me": mu_b_me,
            "N": N_total,
            "N_p001_02": site_counts["p001_02"],
            "N_p008_01": site_counts["p008_01"],
            "N_p011_01": site_counts["p011_01"],
            "N_p011_02": site_counts["p011_02"],
        }
        means_list.append(mean_row)


STEP 2: Calculating Reliability Ratios

--- Processing Outcome 1: Household size ---
  Weight combination: (1.0, 0.0)
  Weight combination: (0.9, 0.1)
  Weight combination: (0.5, 0.5)
  Weight combination: (0.1, 0.9)
  Weight combination: (0.0, 1.0)

--- Processing Outcome 2: Land size (ha) ---
  Weight combination: (1.0, 0.0)
    ERROR fitting model for k=2, i=100: Singular matrix
  Weight combination: (0.9, 0.1)
    ERROR fitting model for k=2, i=90: Singular matrix
  Weight combination: (0.5, 0.5)
    ERROR fitting model for k=2, i=50: Singular matrix
  Weight combination: (0.1, 0.9)
    ERROR fitting model for k=2, i=10: Singular matrix
  Weight combination: (0.0, 1.0)
    ERROR fitting model for k=2, i=0: Singular matrix

--- Processing Outcome 3: No. of crop types ---
  Weight combination: (1.0, 0.0)
    ERROR fitting model for k=3, i=100: Singular matrix
  Weight combination: (0.9, 0.1)
    ERROR fitting model for k=3, i=90: Singular matrix
  Weight combination: (0.5, 0.5)
    

7. CREATE DATAFRAMES FROM RESULTS

In [27]:
results_df = pd.DataFrame(results_list)
means_df = pd.DataFrame(means_list)

print("\n" + "=" * 80)
print("RESULTS SUMMARY")
print("=" * 80)
print(results_df.head(10))


RESULTS SUMMARY
           outcome  weight_s  weight_b     var_s    var_b  var_signal  \
0   Household size       1.0       0.0  0.998185  0.99830    0.998185   
1   Household size       0.9       0.1  0.998185  0.99830    0.974503   
2   Household size       0.5       0.5  0.998185  0.99830    0.932427   
3   Household size       0.1       0.9  0.998185  0.99830    0.974595   
4   Household size       0.0       1.0  0.998185  0.99830    0.998300   
5  Pooled outcomes       1.0       0.0  1.012697  0.98692    1.012697   
6  Pooled outcomes       0.9       0.1  1.012697  0.98692    0.970829   
7  Pooled outcomes       0.5       0.5  1.012697  0.98692    0.890667   
8  Pooled outcomes       0.1       0.9  1.012697  0.98692    0.950207   
9  Pooled outcomes       0.0       1.0  1.012697  0.98692    0.986920   

   var_s_me  var_b_me   rr_ky_s   rr_ky_b   rr_as_s   rr_as_b  iic     N  \
0  0.010171  0.270953  0.989913  0.786506  0.989811  0.728585  0.0  1964   
1  0.013155  0.221751  0.98

8. DISPLAY RESULTS IN JUPYTER NOTEBOOK

In [28]:
print("\n" + "=" * 80)
print("STEP 3: Displaying Results")
print("=" * 80)

# Format numeric columns for better display
numeric_cols_var = [
    "var_s",
    "var_b",
    "var_signal",
    "var_s_me",
    "var_b_me",
    "rr_ky_s",
    "rr_ky_b",
    "rr_as_s",
    "rr_as_b",
    "iic",
]
numeric_cols_mean = ["mu_s", "mu_b", "mu_signal", "mu_s_me", "mu_b_me"]

# Create styled display DataFrames
results_display = results_df.copy()
means_display = means_df.copy()

# Round numeric columns
for col in numeric_cols_var:
    if col in results_display.columns:
        results_display[col] = results_display[col].round(3)

for col in numeric_cols_mean:
    if col in means_display.columns:
        means_display[col] = means_display[col].round(3)

# Round weight columns
results_display["weight_s"] = results_display["weight_s"].round(1)
results_display["weight_b"] = results_display["weight_b"].round(1)
means_display["weight_s"] = means_display["weight_s"].round(1)
means_display["weight_b"] = means_display["weight_b"].round(1)


STEP 3: Displaying Results


VARIANCE TABLE

In [29]:
print("\n" + "=" * 80)
print("RELIABILITY STATISTICS: VARIANCE TABLE (Abowd & Stinson Table 6)")
print("=" * 80 + "\n")

# Display with styling


# Style function for better visualization
def style_dataframe(df, title):
    """Apply styling to dataframe for better display"""
    styled = (
        df.style.set_properties(**{"text-align": "center", "font-size": "11pt"})
        .set_table_styles(
            [
                {
                    "selector": "th",
                    "props": [
                        ("background-color", "#4CAF50"),
                        ("color", "white"),
                        ("font-weight", "bold"),
                        ("text-align", "center"),
                        ("font-size", "11pt"),
                    ],
                },
                {"selector": "td", "props": [("padding", "8px")]},
                {"selector": "tr:hover", "props": [("background-color", "#f5f5f5")]},
            ]
        )
        .set_caption(title)
    )

    # Highlight reliability ratios
    styled = styled.background_gradient(
        subset=["rr_ky_s", "rr_ky_b", "rr_as_s", "rr_as_b"],
        cmap="RdYlGn",
        vmin=0,
        vmax=1,
    )

    # Highlight IIC
    styled = styled.background_gradient(subset=["iic"], cmap="YlOrRd", vmin=0, vmax=1)

    return styled


# Display variance table with styling
display(
    style_dataframe(
        results_display,
        "Reliability Statistics: Variance Components and Reliability Ratios",
    )
)


RELIABILITY STATISTICS: VARIANCE TABLE (Abowd & Stinson Table 6)



,outcome,weight_s,weight_b,var_s,var_b,var_signal,var_s_me,var_b_me,rr_ky_s,rr_ky_b,rr_as_s,rr_as_b,iic,N,N_p001_02,N_p008_01,N_p011_01,N_p011_02
0,Household size,1.000000,0.000000,0.998000,0.998000,0.998000,0.010000,0.271000,0.990000,0.787000,0.990000,0.729000,0.000000,1964,612,171,591,590
1,Household size,0.900000,0.100000,0.998000,0.998000,0.975000,0.013000,0.222000,0.987000,0.815000,0.987000,0.778000,0.000000,1964,612,171,591,590
2,Household size,0.500000,0.500000,0.998000,0.998000,0.932000,0.077000,0.077000,0.923000,0.924000,0.923000,0.923000,0.000000,1964,612,171,591,590
3,Household size,0.100000,0.900000,0.998000,0.998000,0.975000,0.225000,0.016000,0.812000,0.984000,0.774000,0.984000,0.000000,1964,612,171,591,590
4,Household size,0.000000,1.000000,0.998000,0.998000,0.998000,0.275000,0.014000,0.784000,0.986000,0.724000,0.986000,0.000000,1964,612,171,591,590
5,Pooled outcomes,1.000000,0.000000,1.013000,0.987000,1.013000,0.000000,0.435000,1.000000,0.700000,1.000000,0.560000,0.000000,5366,1224,675,1714,1753
6,Pooled outcomes,0.900000,0.100000,1.013000,0.987000,0.971000,0.004000,0.352000,0.996000,0.734000,0.996000,0.643000,0.000000,5366,1224,675,1714,1753
7,Pooled outcomes,0.500000,0.500000,1.013000,0.987000,0.891000,0.109000,0.109000,0.891000,0.891000,0.893000,0.890000,0.000000,5366,1224,675,1714,1753
8,Pooled outcomes,0.100000,0.900000,1.013000,0.987000,0.950000,0.352000,0.004000,0.730000,0.995000,0.652000,0.996000,0.000000,5366,1224,675,1714,1753
9,Pooled outcomes,0.000000,1.000000,1.013000,0.987000,0.987000,0.435000,0.000000,0.694000,1.000000,0.571000,1.000000,0.000000,5366,1224,675,1714,1753


MEANS TABLE

In [30]:
print("\n" + "=" * 80)
print("ADDITIONAL INFO: MEANS TABLE")
print("=" * 80 + "\n")

# Display means table with styling
styled_means = (
    means_display.style.set_properties(**{"text-align": "center", "font-size": "11pt"})
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("background-color", "#2196F3"),
                    ("color", "white"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                    ("font-size", "11pt"),
                ],
            },
            {"selector": "td", "props": [("padding", "8px")]},
            {"selector": "tr:hover", "props": [("background-color", "#f5f5f5")]},
        ]
    )
    .set_caption("Mean Values of Outcomes and Measurement Errors")
)

display(styled_means)


ADDITIONAL INFO: MEANS TABLE



,outcome,weight_s,weight_b,mu_s,mu_b,mu_signal,mu_s_me,mu_b_me,N,N_p001_02,N_p008_01,N_p011_01,N_p011_02
0,Household size,1.000000,0.000000,-0.030000,0.029000,-0.030000,0.314000,0.501000,1964,612,171,591,590
1,Household size,0.900000,0.100000,-0.030000,0.029000,-0.024000,0.316000,0.503000,1964,612,171,591,590
2,Household size,0.500000,0.500000,-0.030000,0.029000,-0.000000,0.322000,0.509000,1964,612,171,591,590
3,Household size,0.100000,0.900000,-0.030000,0.029000,0.023000,0.329000,0.516000,1964,612,171,591,590
4,Household size,0.000000,1.000000,-0.030000,0.029000,0.029000,0.330000,0.517000,1964,612,171,591,590
5,Pooled outcomes,1.000000,0.000000,-0.027000,0.030000,-0.027000,0.000000,0.176000,5366,1224,675,1714,1753
6,Pooled outcomes,0.900000,0.100000,-0.027000,0.030000,-0.021000,-0.018000,0.158000,5366,1224,675,1714,1753
7,Pooled outcomes,0.500000,0.500000,-0.027000,0.030000,0.002000,-0.088000,0.088000,5366,1224,675,1714,1753
8,Pooled outcomes,0.100000,0.900000,-0.027000,0.030000,0.025000,-0.158000,0.018000,5366,1224,675,1714,1753
9,Pooled outcomes,0.000000,1.000000,-0.027000,0.030000,0.030000,-0.176000,0.000000,5366,1224,675,1714,1753


SUMMARY STATISTICS BY OUTCOME

In [31]:
print("\n" + "=" * 80)
print("SUMMARY: RELIABILITY RATIOS BY OUTCOME (Averaged Across Weights)")
print("=" * 80 + "\n")

# Calculate summary statistics grouped by outcome
summary_stats = (
    results_df.groupby("outcome")
    .agg(
        {
            "rr_ky_s": ["mean", "std", "min", "max"],
            "rr_ky_b": ["mean", "std", "min", "max"],
            "rr_as_s": ["mean", "std", "min", "max"],
            "rr_as_b": ["mean", "std", "min", "max"],
            "iic": ["mean", "std", "min", "max"],
        }
    )
    .round(3)
)

# Flatten column names
summary_stats.columns = ["_".join(col) for col in summary_stats.columns]
summary_stats = summary_stats.reset_index()

# Display summary
styled_summary = (
    summary_stats.style.set_properties(**{"text-align": "center", "font-size": "11pt"})
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("background-color", "#FF9800"),
                    ("color", "white"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                    ("font-size", "11pt"),
                ],
            }
        ]
    )
    .set_caption("Summary Statistics by Outcome Variable")
)

display(styled_summary)


SUMMARY: RELIABILITY RATIOS BY OUTCOME (Averaged Across Weights)



,outcome,rr_ky_s_mean,rr_ky_s_std,rr_ky_s_min,rr_ky_s_max,rr_ky_b_mean,rr_ky_b_std,rr_ky_b_min,rr_ky_b_max,rr_as_s_mean,rr_as_s_std,rr_as_s_min,rr_as_s_max,rr_as_b_mean,rr_as_b_std,rr_as_b_min,rr_as_b_max,iic_mean,iic_std,iic_min,iic_max
0,Household size,0.899000,0.097000,0.784000,0.990000,0.899000,0.094000,0.787000,0.986000,0.880000,0.123000,0.724000,0.990000,0.880000,0.120000,0.729000,0.986000,0.000000,0.000000,0.000000,0.000000
1,Pooled outcomes,0.862000,0.144000,0.694000,1.000000,0.864000,0.142000,0.700000,1.000000,0.822000,0.199000,0.571000,1.000000,0.818000,0.204000,0.560000,1.000000,0.000000,0.000000,0.000000,0.000000


KEY INSIGHTS

In [32]:
print("\n" + "=" * 80)
print("KEY INSIGHTS")
print("=" * 80 + "\n")

# Calculate overall averages
avg_rr_as_s = results_df["rr_as_s"].mean()
avg_rr_as_b = results_df["rr_as_b"].mean()
avg_iic = results_df["iic"].mean()

print(f"Overall Average Reliability Ratio (Survey, AS method):    {avg_rr_as_s:.3f}")
print(f"Overall Average Reliability Ratio (Backcheck, AS method): {avg_rr_as_b:.3f}")
print(f"Overall Average IIC (Interviewer Effect):                 {avg_iic:.3f}")

# Identify outcomes with lowest reliability
worst_outcome = results_df.groupby("outcome")["rr_as_s"].mean().idxmin()
best_outcome = results_df.groupby("outcome")["rr_as_s"].mean().idxmax()

print(f"\nOutcome with lowest reliability:  {worst_outcome}")
print(f"Outcome with highest reliability: {best_outcome}")

# IIC interpretation
print("\n" + "-" * 80)
print("INTERPRETATION NOTES:")
print("-" * 80)
print("\nReliability Ratios (RR):")
print("  • RR close to 1.0 = High data quality (low measurement error)")
print("  • RR close to 0.0 = Low data quality (high measurement error)")
print("\nIntra-class Correlation (IIC):")
print("  • IIC close to 1.0 = Systematic interviewer effects (concerning)")
print("  • IIC close to 0.0 = Random measurement error (less concerning)")
print("-" * 80)

print("\nDone! ✓")

# Store results for further analysis
print("\n" + "=" * 80)
print("DataFrames available for further analysis:")
print("  • results_df: Full variance and reliability ratio results")
print("  • means_df: Mean values for all variables")
print("  • summary_stats: Aggregated statistics by outcome")
print("=" * 80)


KEY INSIGHTS

Overall Average Reliability Ratio (Survey, AS method):    0.851
Overall Average Reliability Ratio (Backcheck, AS method): 0.849
Overall Average IIC (Interviewer Effect):                 0.000

Outcome with lowest reliability:  Pooled outcomes
Outcome with highest reliability: Household size

--------------------------------------------------------------------------------
INTERPRETATION NOTES:
--------------------------------------------------------------------------------

Reliability Ratios (RR):
  • RR close to 1.0 = High data quality (low measurement error)
  • RR close to 0.0 = Low data quality (high measurement error)

Intra-class Correlation (IIC):
  • IIC close to 1.0 = Systematic interviewer effects (concerning)
  • IIC close to 0.0 = Random measurement error (less concerning)
--------------------------------------------------------------------------------

Done! ✓

DataFrames available for further analysis:
  • results_df: Full variance and reliability ratio res